In [2]:
import pandas as pd
import numpy as np

In [3]:
movies = pd.read_csv('ml-32m/movies.csv')
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies.shape

(87585, 3)

In [5]:
#filtred_movies = movies[movies.movieId<10000]
filtred_movies = movies

In [6]:
ratings = pd.read_csv('ml-32m/ratings.csv')
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [7]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 976.6 MB


In [8]:
len(ratings.userId.unique())

200948

In [9]:
max(ratings.movieId)

292757

In [10]:
merged_data = pd.merge(ratings, filtred_movies[['movieId', 'title']], on='movieId')

In [11]:
merged_data.shape

(32000204, 5)

In [12]:
merged_data.userId= merged_data.userId.astype(np.int32)

In [13]:
merged_data.movieId = merged_data.movieId.astype(np.int32)

In [14]:
merged_data.rating = merged_data.rating.round().astype(np.int32)

In [15]:
merged_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32000204 entries, 0 to 32000203
Data columns (total 5 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   userId     int32 
 1   movieId    int32 
 2   rating     int32 
 3   timestamp  int64 
 4   title      object
dtypes: int32(3), int64(1), object(1)
memory usage: 854.5+ MB


In [16]:
grouped = merged_data.groupby(['userId', 'title'], observed=True)['rating'].mean().astype(np.int32)

In [17]:
user_item_matrix = grouped.unstack('title', fill_value=0)

C:\Users\tir1\AppData\Local\Temp\ipykernel_6200\1923007970.py:1: PerformanceWarning: The following operation may generate 16927658572 cells in the resulting pandas object.
  user_item_matrix = grouped.unstack('title', fill_value=0)


In [ ]:
#user_item_matrix = merged_data.pivot_table(index='userId', columns='title', values='rating', fill_value=0)

In [18]:
user_item_matrix.shape

(200948, 84239)

In [19]:
user_item_matrix

title,(2019),"""BLOW THE NIGHT!"" Let's Spend the Night Together (1983)","""Great Performances"" Cats (1998)","""Sr."" (2022)",#1 Cheerleader Camp (2010),#Alive (2020),#AnneFrank. Parallel Stories (2019),#Captured (2017),#Female Pleasure (2018),#FollowMe (2019),...,…And the Fifth Horseman Is Fear (1965),キサラギ (2007),チェブラーシカ (2010),ユニコ 魔法の島へ (1983),他们低语 (2021),牛づれ超特急 (1937),茶叶之旅 (2014),貞子3D (2012),过昭关,줄탁동시 (2012)
userId,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200944,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
200945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
200946,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
user_item_matrix = user_item_matrix.loc[user_item_matrix.astype(bool).sum(axis=1) >= 70, user_item_matrix.astype(bool).sum(axis=0) >= 1000]

In [21]:
import re

# Заданный год для фильтрации
year_threshold = 1980

# Создаем список столбцов для удаления
cols_to_drop = []
for column in user_item_matrix.columns[1:]:  # Пропускаем первый столбец 'userId'
    # Ищем год в формате (XXXX) в конце названия
    match = re.search(r'\((\d{4})\)$', column)
    if match:
        year = int(match.group(1))
        if year < year_threshold:
            cols_to_drop.append(column)
    else:
        print(f"Предупреждение: в столбце '{column}' не найден год.")

# Удаляем столбцы, не соответствующие условию
user_item_matrix = user_item_matrix.drop(columns=cols_to_drop)

# Проверяем сохранение столбца 'userId'
if 'userId' not in user_item_matrix.columns:
    print("Ошибка: столбец 'userId' был удален.")
else:
    print("Фильтрация завершена успешно.")

Предупреждение: в столбце '13 reasons why' не найден год.
Предупреждение: в столбце 'Angel Has Fallen' не найден год.
Предупреждение: в столбце 'Black Mirror' не найден год.
Предупреждение: в столбце 'Code 8' не найден год.
Предупреждение: в столбце 'Cosmos' не найден год.
Предупреждение: в столбце 'Cosmos: A Spacetime Odissey' не найден год.
Предупреждение: в столбце 'Death Note: Desu nôto (2006–2007)' не найден год.
Предупреждение: в столбце 'Enola Holmes' не найден год.
Предупреждение: в столбце 'I'm Thinking of Ending Things' не найден год.
Предупреждение: в столбце 'In the Tall Grass' не найден год.
Предупреждение: в столбце 'Moonlight' не найден год.
Предупреждение: в столбце 'Nocturnal Animals' не найден год.
Предупреждение: в столбце 'Paterson' не найден год.
Предупреждение: в столбце 'Ready Player One' не найден год.
Предупреждение: в столбце 'The Favourite' не найден год.
Предупреждение: в столбце 'The Green Knight' не найден год.
Предупреждение: в столбце 'The King' не найде

In [25]:
user_item_matrix.shape

(94342, 3735)

In [39]:
user_item_matrix.head(20)

title,'71 (2014),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),10 Cloverfield Lane (2016),10 Items or Less (2006),10 Things I Hate About You (1999),"10,000 BC (2008)",...,Zootopia (2016),"Zorro, the Gay Blade (1981)",[REC] (2007),[REC]² (2009),eXistenZ (1999),"tick, tick...BOOM! (2021)",xXx (2002),xXx: Return of Xander Cage (2017),xXx: State of the Union (2005),¡Three Amigos! (1986)
userId,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,2,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
user_item_matrix.to_csv('UI_data_2.csv')

In [27]:
fill_rate = user_item_matrix.astype(bool).sum().sum() / user_item_matrix.size
fill_rate_percentage = fill_rate * 100
print(f"Заполненность матрицы: {fill_rate_percentage:.2f}%")

Заполненность матрицы: 6.03%
